In [1]:
# pip install pymupdf sentence-transformers langchain-text-splitters pymongo

import fitz
import re
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pymongo import MongoClient

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
LAW_CONFIG = {

    "BNS": {
        "pdf_path": r"./statutes/New/bns/THE_BHARATIYA_NYAYA_SANHITA_2023.pdf",
        "max_section": 358,
        "schedule_marker": None
    },

    "BNSS": {
        "pdf_path": r"./statutes/New/bnss/THE_BHARATIYA_NAGARIK_SURAKSHA_SANHITA_2023.pdf",
        "max_section": 531,
        "schedule_marker": "THE FIRST SCHEDULE"
    },

    "BSA": {
        "pdf_path": r"./statutes/New/bsa/THE_BHARATIYA_SAKSHYA_ADHINIYAM_2023.pdf",
        "max_section": 170,
        "schedule_marker": "THE SCHEDULE"
    }
}

In [8]:
def extract_main_text(page):

    blocks = page.get_text("blocks")

    kept_blocks = []

    for block in blocks:

        x0, y0, x1, y1, text, *_ = block

        # Main legal-text column
        # Removes most marginal headings and Gazette headers
        if (
            x0 >= 110
            and x1 <= 490
            and 80 <= y0 <= 780
        ):

            text = text.strip()

            if not text:
                continue

            upper = text.upper()

            # Extra protection
            if "THE GAZETTE OF INDIA" in upper:
                continue

            if "[PART II" in upper:
                continue

            kept_blocks.append(
                (y0, x0, text)
            )

    # Restore reading order
    kept_blocks.sort(
        key=lambda x: (x[0], x[1])
    )

    return "\n".join(
        text
        for _, _, text in kept_blocks
    )

In [9]:
def extract_pdf(pdf_path):

    pdf = fitz.open(pdf_path)

    full_text = ""

    for page_number, page in enumerate(
        pdf,
        start=1
    ):

        text = extract_main_text(page)

        full_text += (
            f"\n<<<PAGE:{page_number}>>>\n"
            + text
            + "\n"
        )

    return full_text

In [10]:
def parse_sections(
    pdf_path,
    law,
    max_section,
    schedule_marker=None
):

    full_text = extract_pdf(
        pdf_path
    )

    # -------------------------
    # Remove schedules
    # -------------------------

    if schedule_marker:

        marker_position = full_text.find(
            schedule_marker
        )

        if marker_position != -1:
            full_text = full_text[
                :marker_position
            ]

    # -------------------------
    # Find sections sequentially
    # -------------------------

    matches = []

    search_position = 0

    for expected_section in range(
        1,
        max_section + 1
    ):

        pattern = re.compile(
            rf"(?m)^\s*{expected_section}\.\s*"
        )

        match = pattern.search(
            full_text,
            search_position
        )

        if match is None:

            raise ValueError(
                f"{law}: Section "
                f"{expected_section} not found."
            )

        matches.append(
            (
                expected_section,
                match
            )
        )

        search_position = match.end()

    # -------------------------
    # Create section documents
    # -------------------------

    sections = []

    for i, (
        section_number,
        match
    ) in enumerate(matches):

        start = match.start()

        if i + 1 < len(matches):

            end = matches[
                i + 1
            ][1].start()

        else:

            end = len(full_text)

        raw_section = full_text[
            start:end
        ]

        # Find source pages
        page_numbers = [
            int(page)
            for page in re.findall(
                r"<<<PAGE:(\d+)>>>",
                raw_section
            )
        ]

        # Remove internal page markers
        section_text = re.sub(
            r"<<<PAGE:\d+>>>",
            "",
            raw_section
        ).strip()

        sections.append(
            Document(
                page_content=section_text,

                metadata={
                    "law": law,

                    "section":
                        str(section_number),

                    "pages":
                        sorted(
                            set(page_numbers)
                        ),

                    "source":
                        pdf_path,

                    "document_type":
                        "primary_statute"
                }
            )
        )

    return sections

In [11]:
all_sections = {}

for law, config in LAW_CONFIG.items():

    sections = parse_sections(

        pdf_path=
            config["pdf_path"],

        law=law,

        max_section=
            config["max_section"],

        schedule_marker=
            config["schedule_marker"]
    )

    all_sections[law] = sections

    print(
        law,
        "sections:",
        len(sections)
    )

BNS sections: 358
BNSS sections: 531
BSA sections: 170


In [12]:
def validate_sections(
    sections,
    max_section
):

    actual = [
        int(doc.metadata["section"])
        for doc in sections
    ]

    expected = list(
        range(
            1,
            max_section + 1
        )
    )

    if actual == expected:

        print(
            f"PASS: All {max_section} "
            "sections found correctly."
        )

        return True

    print("FAIL")

    missing = sorted(
        set(expected)
        - set(actual)
    )

    duplicates = [
        n
        for n in actual
        if actual.count(n) > 1
    ]

    print(
        "Missing:",
        missing
    )

    print(
        "Duplicates:",
        sorted(set(duplicates))
    )

    return False

In [13]:
for law, config in LAW_CONFIG.items():

    validate_sections(
        all_sections[law],
        config["max_section"]
    )

PASS: All 358 sections found correctly.
PASS: All 531 sections found correctly.
PASS: All 170 sections found correctly.


In [14]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Model max length:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1355.30it/s]


Model max length: 256


In [15]:
splitter = (
    RecursiveCharacterTextSplitter
    .from_huggingface_tokenizer(

        tokenizer=model.tokenizer,

        chunk_size=200,

        chunk_overlap=30,

        separators=[
            "\n\n",
            "\n(",
            "\nExplanation",
            "\nIllustration",
            "\nProvided",
            ". ",
            " "
        ]
    )
)

In [16]:
def create_chunks(
    sections,
    model,
    splitter
):

    chunks = []

    for section in sections:

        text = section.page_content

        token_count = len(
            model.tokenizer.encode(
                text,
                add_special_tokens=False
            )
        )

        # -------------------
        # Small section
        # -------------------

        if token_count <= 200:

            chunks.append(
                Document(
                    page_content=text,

                    metadata={
                        **section.metadata,

                        "parent_section":
                            section.metadata[
                                "section"
                            ],

                        "chunk_index": 0
                    }
                )
            )

        # -------------------
        # Large section
        # -------------------

        else:

            parts = splitter.split_text(
                text
            )

            for index, part in enumerate(
                parts
            ):

                chunks.append(
                    Document(
                        page_content=part,

                        metadata={
                            **section.metadata,

                            "parent_section":
                                section.metadata[
                                    "section"
                                ],

                            "chunk_index":
                                index
                        }
                    )
                )

    return chunks

In [17]:
all_chunks = []

for law in LAW_CONFIG:

    law_chunks = create_chunks(

        sections=
            all_sections[law],

        model=model,

        splitter=splitter
    )

    print(
        law,
        "chunks:",
        len(law_chunks)
    )

    all_chunks.extend(
        law_chunks
    )


print(
    "\nTotal chunks:",
    len(all_chunks)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (355 > 256). Running this sequence through the model will result in indexing errors


BNS chunks: 637
BNSS chunks: 946
BSA chunks: 271

Total chunks: 1854
